<a href="https://colab.research.google.com/github/77MJohn77/Geospatial-Research-Portfolio/blob/main/SDG_15_3_1_Kayonza_LandDegradation_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌍 SDG Indicator 15.3.1 – Land Degradation Analysis
## Kayonza District, Rwanda

**GEE Project:** `dotted-music-307621`  
**AOI Asset:** `projects/dotted-music-307621/assets/KAYONZA`  
**Methodology:** UNCCD Good Practice Guidance (GPG) for SDG 15.3.1 *(Trends.Earth approach)*

### Sub-indicators
| # | Sub-indicator | Data |
|---|---|---|
| 1 | **Land Productivity** | Trajectory + State + Performance (MODIS NDVI) |
| 2 | **Land Cover Change** | ESA CCI → 7 UNCCD classes |
| 3 | **Soil Organic Carbon** | SoilGrids + IPCC land-use coefficients |

**Final indicator:** One-Out All-Out (1OAO) rule — if *any* sub-indicator signals degradation, the pixel is classified as degraded.

---
> ⚠️ **Run cells sequentially.** Authenticate GEE when prompted in Cell 1.

## Cell 1 – Install & Authenticate

In [2]:
!pip install -q earthengine-api geemap

import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

# Authenticate & initialise with your GEE project
ee.Authenticate()
ee.Initialize(project='dotted-music-307621')
print("Earth Engine initialised ✓")

Earth Engine initialised ✓


## Cell 2 – Area of Interest (Kayonza)

In [3]:
AOI      = ee.FeatureCollection('projects/dotted-music-307621/assets/KAYONZA')
aoi_geom = AOI.geometry()

print("AOI loaded:", AOI.size().getInfo(), "feature(s)")

AOI loaded: 1 feature(s)


## Cell 3 – Global Parameters

Adjust `TARGET_END` to the latest available MODIS year if needed.

In [4]:
# ── Analysis periods (align with UNCCD reporting) ───────────────────────────
BASELINE_START = 2001   # MODIS era start
BASELINE_END   = 2015   # SDG baseline year
TARGET_START   = 2016
TARGET_END     = 2023   # adjust to latest available year

BASELINE_YEARS = list(range(BASELINE_START, BASELINE_END + 1))
TARGET_YEARS   = list(range(TARGET_START,   TARGET_END   + 1))
ALL_YEARS      = list(range(BASELINE_START, TARGET_END   + 1))

# ── Thresholds ───────────────────────────────────────────────────────────────
P_VALUE_THRESH   = 0.05   # significance for Mann-Kendall (proxy)
PERF_THRESHOLD   = 0.50   # < 50 % of 90th-pct NDVI → degraded
SOC_THRESHOLD    = 0.10   # ±10 % SOC change threshold
TRAJ_STABLE_BAND = 0.001  # NDVI/year treated as stable

# Rwanda = Tropical Moist → f = 0.48 (UNCCD/IPCC)
F_COEFF = 0.48

print(f"Baseline  : {BASELINE_START}–{BASELINE_END}  ({len(BASELINE_YEARS)} years)")
print(f"Target    : {TARGET_START}–{TARGET_END}  ({len(TARGET_YEARS)} years)")
print(f"All years : {len(ALL_YEARS)}")

Baseline  : 2001–2015  (15 years)
Target    : 2016–2023  (8 years)
All years : 23


## Cell 4 – Annual NDVI Collection (MODIS MOD13Q1, 250 m)

Builds three `ImageCollection` objects: full period, baseline, and target.

In [5]:
def get_annual_ndvi(year):
    """Return mean annual NDVI image for a given year (MODIS 250 m)."""
    start = ee.Date.fromYMD(year, 1, 1)
    end   = ee.Date.fromYMD(year, 12, 31)
    col   = (ee.ImageCollection('MODIS/061/MOD13Q1')
             .filterDate(start, end)
             .filterBounds(aoi_geom)
             .select('NDVI'))
    return (col.mean()
               .multiply(0.0001)   # scale factor → [-1, 1]
               .rename('NDVI')
               .set('year', year)
               .clip(aoi_geom))

ndvi_all      = ee.ImageCollection([get_annual_ndvi(y) for y in ALL_YEARS])
ndvi_baseline = ee.ImageCollection([get_annual_ndvi(y) for y in BASELINE_YEARS])
ndvi_target   = ee.ImageCollection([get_annual_ndvi(y) for y in TARGET_YEARS])

print(f"NDVI collection: {len(ALL_YEARS)} annual images ({BASELINE_START}–{TARGET_END})")

NDVI collection: 23 annual images (2001–2023)


## Cell 5 – Productivity Trajectory

Linear regression (Sen's slope proxy via GEE `linearFit`) across all years.  
- Slope > +0.001 NDVI/yr → **Improving**  
- Slope < −0.001 NDVI/yr → **Degrading**  
- Otherwise → **Stable**

In [6]:
def add_time_band(image):
    year = image.date().get('year')
    time = ee.Image(ee.Number(year)).float().rename('time')
    return image.addBands(time)

ndvi_with_time = ndvi_all.map(add_time_band).select(['time', 'NDVI'])

# Pixel-level linear regression
lr    = ndvi_with_time.reduce(ee.Reducer.linearFit()).clip(aoi_geom)
slope = lr.select('scale').rename('slope')

# Classify
traj_improving = slope.gt(TRAJ_STABLE_BAND)
traj_degrading = slope.lt(-TRAJ_STABLE_BAND)

trajectory = (traj_improving.multiply(1)
              .add(traj_degrading.multiply(-1))
              .rename('trajectory'))

print("Trajectory computed ✓  (1=Improving | 0=Stable | -1=Degrading)")

Trajectory computed ✓  (1=Improving | 0=Stable | -1=Degrading)


## Cell 6 – Productivity State

Compares mean NDVI of the last 3 target years to the baseline distribution.  
Pixels shifting ≥ 2 percentile classes downward → **Degrading**; upward → **Improving**.

In [7]:
ndvi_mean_baseline = ndvi_baseline.mean().rename('ndvi_base').clip(aoi_geom)
ndvi_comparison    = (ee.ImageCollection([get_annual_ndvi(y) for y in TARGET_YEARS[-3:]])
                      .mean().rename('ndvi_comp').clip(aoi_geom))

# Baseline percentile thresholds (10th … 90th)
percentiles = ndvi_baseline.reduce(
    ee.Reducer.percentile([10, 20, 30, 40, 50, 60, 70, 80, 90])
).clip(aoi_geom)

def assign_class(image, pcts, prefix=''):
    """Assign each pixel to one of 10 percentile classes."""
    p = [pcts.select(f'NDVI_p{v}') for v in [10,20,30,40,50,60,70,80,90]]
    cls = (image.lt(p[0]).multiply(1)
           .add(image.gte(p[0]).And(image.lt(p[1])).multiply(2))
           .add(image.gte(p[1]).And(image.lt(p[2])).multiply(3))
           .add(image.gte(p[2]).And(image.lt(p[3])).multiply(4))
           .add(image.gte(p[3]).And(image.lt(p[4])).multiply(5))
           .add(image.gte(p[4]).And(image.lt(p[5])).multiply(6))
           .add(image.gte(p[5]).And(image.lt(p[6])).multiply(7))
           .add(image.gte(p[6]).And(image.lt(p[7])).multiply(8))
           .add(image.gte(p[7]).And(image.lt(p[8])).multiply(9))
           .add(image.gte(p[8]).multiply(10)))
    return cls.rename(f'{prefix}class')

base_class = assign_class(ndvi_mean_baseline, percentiles, 'base_')
comp_class = assign_class(ndvi_comparison,    percentiles, 'comp_')
class_diff = comp_class.subtract(base_class).rename('class_diff')

state = (class_diff.gte(2).multiply(1)
         .add(class_diff.lte(-2).multiply(-1))
         .rename('state'))

print("State computed ✓")

State computed ✓


## Cell 7 – Productivity Performance

Compares each pixel's mean NDVI to the 90th percentile of its eco-unit  
(unique combination of ESA CCI land cover × SoilGrids USDA taxonomy).  
- Ratio < 50 % → **Degrading**  
- Ratio ≥ 80 % → **Improving**

In [8]:
# ESA CCI Land Cover 2015 (representative baseline year)
esa_lc_2015 = (ee.ImageCollection('ESA/CCI/FireCCI/5_1')
               .filterDate('2015-01-01', '2015-12-31')
               .first()
               .clip(aoi_geom))

# SoilGrids USDA taxonomy (static)
soil = (ee.Image('OpenLandMap/SOL/SOL_TAXOUSDA_USDA-ST_M/v01')
        .select('b0')
        .clip(aoi_geom))

# Unique eco-unit key
lc_band   = esa_lc_2015.select([0]).rename('lc').uint16()
soil_band = soil.rename('soil').uint16()
eco_unit  = lc_band.multiply(1000).add(soil_band).rename('eco_unit')

# Mean NDVI over the full analysis period
ndvi_mean_all = ndvi_all.mean().rename('NDVI').clip(aoi_geom)

# 90th-percentile NDVI per eco-unit (connected components)
p90_per_unit = (ndvi_mean_all.addBands(eco_unit)
                .reduceConnectedComponents(
                    reducer=ee.Reducer.percentile([90]),
                    labelBand='eco_unit')
                .rename('p90_ndvi'))

ratio = ndvi_mean_all.divide(p90_per_unit).rename('perf_ratio')

performance = (ratio.gte(0.80).multiply(1)
               .add(ratio.lt(PERF_THRESHOLD).multiply(-1))
               .rename('performance'))

print("Performance computed ✓")

Performance computed ✓


## Cell 8 – Combine Productivity Metrics (1OAO → 3-class)

| Trajectory | State | Performance | → Productivity |
|---|---|---|---|
| Any = Degrading | | | **Degrading** |
| All = Improving | | | **Improving** |
| Otherwise | | | **Stable** |

In [9]:
any_degrading_prod = (trajectory.eq(-1)
                      .Or(state.eq(-1))
                      .Or(performance.eq(-1)))

all_improving_prod = (trajectory.eq(1)
                      .And(state.eq(1))
                      .And(performance.eq(1)))

productivity = (all_improving_prod.multiply(1)
                .add(any_degrading_prod.multiply(-1))
                .rename('productivity'))

print("Combined productivity computed ✓")

Combined productivity computed ✓


## Cell 9 – Land Cover Sub-indicator

ESA CCI reclassified to 7 UNCCD classes, then the UNCCD degradation  
transition matrix is applied to identify improving / stable / degrading pixels.

| UNCCD Class | Code |
|---|---|
| Forest | 1 |
| Grassland | 2 |
| Cropland | 3 |
| Wetland | 4 |
| Artificial / Settlement | 5 |
| Bare land | 6 |
| Water body | 7 |

In [10]:
def reclassify_esa_to_unccd(lc_image):
    """Remap ESA CCI codes → 7 UNCCD classes."""
    from_vals = ee.List([
        10,20,30,40,                           # Cropland → 3
        50,60,61,62,70,71,72,80,81,82,90,100,  # Forest → 1
        110,120,121,122,130,140,               # Grassland → 2
        150,151,152,153,                       # Bare → 6
        160,170,180,                           # Wetland → 4
        190,                                   # Artificial → 5
        200,201,202,                           # Bare → 6
        210,                                   # Water → 7
        220                                    # Snow/ice → 6
    ])
    to_vals = ee.List([
        3,3,3,3,
        1,1,1,1,1,1,1,1,1,1,1,1,
        2,2,2,2,2,2,
        6,6,6,6,
        4,4,4,
        5,
        6,6,6,
        7,
        6
    ])
    return lc_image.remap(from_vals, to_vals, defaultValue=6).uint8()

# Load ESA CCI for baseline and target years
esa_baseline_img = (ee.ImageCollection('ESA/CCI/FireCCI/5_1')
                    .filterDate(f'{BASELINE_START}-01-01', f'{BASELINE_START}-12-31')
                    .first().clip(aoi_geom))
esa_target_img   = (ee.ImageCollection('ESA/CCI/FireCCI/5_1')
                    .filterDate('2020-01-01', '2020-12-31')
                    .first().clip(aoi_geom))

lc_base   = reclassify_esa_to_unccd(esa_baseline_img.select([0])).rename('lc_base')
lc_target = reclassify_esa_to_unccd(esa_target_img.select([0])).rename('lc_target')

# ── UNCCD transition matrix ────────────────────────────────────────────────
TRANS_MATRIX = {
    (1,1):0,(1,2):-1,(1,3):-1,(1,4):-1,(1,5):-1,(1,6):-1,(1,7):0,
    (2,1):1,(2,2):0,(2,3):-1,(2,4):-1,(2,5):-1,(2,6):-1,(2,7):0,
    (3,1):1,(3,2):1,(3,3):0,(3,4):-1,(3,5):-1,(3,6):-1,(3,7):0,
    (4,1):1,(4,2):1,(4,3):1,(4,4):0,(4,5):-1,(4,6):-1,(4,7):0,
    (5,1):1,(5,2):1,(5,3):1,(5,4):1,(5,5):0,(5,6):-1,(5,7):0,
    (6,1):1,(6,2):1,(6,3):1,(6,4):1,(6,5):-1,(6,6):0,(6,7):0,
    (7,1):0,(7,2):0,(7,3):0,(7,4):0,(7,5):-1,(7,6):-1,(7,7):0,
}

from_codes = [f*10+t for (f,t) in TRANS_MATRIX.keys()]
results    = list(TRANS_MATRIX.values())

transition_key = lc_base.multiply(10).add(lc_target).rename('trans_key')
lc_indicator   = transition_key.remap(from_codes, results, defaultValue=0
                                      ).rename('lc_indicator')

print("Land cover sub-indicator computed ✓")

Land cover sub-indicator computed ✓


## Cell 10 – Soil Organic Carbon Sub-indicator

1. **Reference SOC** from SoilGrids (0–30 cm stock, t C/ha)  
2. **IPCC land-use change coefficients** applied per transition  
   (Tropical Moist zone: *f* = 0.48 for cropland)  
3. Pixels with ≥ 10 % SOC loss → **Degrading**; ≥ 10 % gain → **Improving**

In [11]:
# SoilGrids SOC (0-30 cm) → convert to tonnes C/ha
soc_ref = (ee.Image('OpenLandMap/SOL/SOL_ORGANIC.CARBON_USDA-6A1C_M/v02')
           .select('b0')
           .multiply(0.1)
           .clip(aoi_geom)
           .rename('soc_ref'))

# IPCC land-use coefficients – Tropical Moist (Rwanda)
LU_COEFF = {
    (1,1):1.0,(1,2):1.0,(1,3):F_COEFF,(1,4):1.0,(1,5):0.1,(1,6):0.1,(1,7):1.0,
    (2,1):1.0,(2,2):1.0,(2,3):F_COEFF,(2,4):1.0,(2,5):0.1,(2,6):0.1,(2,7):1.0,
    (3,1):1/F_COEFF,(3,2):1/F_COEFF,(3,3):1.0,(3,4):1/0.71,(3,5):0.1,(3,6):0.1,(3,7):1.0,
    (4,1):1.0,(4,2):1.0,(4,3):0.71,(4,4):1.0,(4,5):0.1,(4,6):0.1,(4,7):1.0,
    (5,1):2.0,(5,2):2.0,(5,3):2.0,(5,4):2.0,(5,5):1.0,(5,6):1.0,(5,7):1.0,
    (6,1):2.0,(6,2):2.0,(6,3):2.0,(6,4):2.0,(6,5):1.0,(6,6):1.0,(6,7):1.0,
    (7,1):1.0,(7,2):1.0,(7,3):1.0,(7,4):1.0,(7,5):1.0,(7,6):1.0,(7,7):1.0,
}

lu_keys   = [k[0]*10+k[1] for k in LU_COEFF.keys()]
lu_coeffs = [round(v, 6) for v in LU_COEFF.values()]

soc_coeff_img = transition_key.remap(lu_keys, lu_coeffs, defaultValue=1.0
                                     ).rename('soc_coeff')
soc_estimated = soc_ref.multiply(soc_coeff_img).rename('soc_estimated')
soc_change    = (soc_estimated.subtract(soc_ref)
                 .divide(soc_ref)
                 .rename('soc_change'))

soc_indicator = (soc_change.gte(SOC_THRESHOLD).multiply(1)
                 .add(soc_change.lte(-SOC_THRESHOLD).multiply(-1))
                 .rename('soc_indicator'))

print("SOC sub-indicator computed ✓")

SOC sub-indicator computed ✓


## Cell 11 – Final SDG 15.3.1 (One-Out All-Out rule)

> **1OAO:** If *any* sub-indicator is Degrading → pixel = **Degraded**  
> If *all* sub-indicators are Improving → pixel = **Improved**  
> Otherwise → **Stable**

In [12]:
any_degrading = (productivity.eq(-1)
                 .Or(lc_indicator.eq(-1))
                 .Or(soc_indicator.eq(-1)))

all_improving = (productivity.eq(1)
                 .And(lc_indicator.eq(1))
                 .And(soc_indicator.eq(1)))

sdg_15_3_1 = (all_improving.multiply(1)
              .add(any_degrading.multiply(-1))
              .rename('SDG_15_3_1'))

# Mask water bodies (UNCCD requirement)
water_mask = lc_base.neq(7).And(lc_target.neq(7))
sdg_15_3_1 = sdg_15_3_1.updateMask(water_mask)

print("SDG 15.3.1 final indicator computed ✓")

SDG 15.3.1 final indicator computed ✓


## Cell 12 – Status Map (Baseline × Period Assessment)

Applies the UNCCD 3×3 Status Matrix:

| | **Degraded** | **Stable** | **Improved** |
|---|---|---|---|
| **Baseline Degraded** | Degraded | Degraded | Improved |
| **Baseline Stable** | Degraded | Stable | Improved |
| **Baseline Improved** | Degraded | Improved | Improved |

In [13]:
# Re-derive baseline SDG indicator using trajectory over baseline period
ndvi_base_traj      = ee.ImageCollection([get_annual_ndvi(y) for y in BASELINE_YEARS])
ndvi_with_time_base = ndvi_base_traj.map(add_time_band).select(['time','NDVI'])
lr_base             = ndvi_with_time_base.reduce(ee.Reducer.linearFit()).clip(aoi_geom)
slope_base          = lr_base.select('scale')

traj_base   = (slope_base.gt(TRAJ_STABLE_BAND).multiply(1)
               .add(slope_base.lt(-TRAJ_STABLE_BAND).multiply(-1))
               .rename('traj_base'))
sdg_baseline = traj_base.rename('sdg_baseline')

# Encode for matrix lookup: shift -1→1, 0→2, 1→3
baseline_enc = sdg_baseline.add(2)
period_enc   = sdg_15_3_1.add(2)
status_key   = baseline_enc.multiply(10).add(period_enc).rename('status_key')

# 3×3 matrix (encoded)
status_from = [11,12,13, 21,22,23, 31,32,33]
status_to   = [-1,-1, 1, -1, 0, 1, -1, 1, 1]   # -1=Deg, 0=Stab, 1=Imp

status_map = status_key.remap(status_from, status_to, defaultValue=0
                              ).rename('status_map')

print("Status map computed ✓")

Status map computed ✓


## Cell 13 – Area Statistics

In [17]:
SCALE = 250   # metres (matches MODIS resolution)

def compute_area_stats(indicator_img, label):
    """Return a DataFrame with area (km²) and % per class. Handles missing/failing assets gracefully."""
    try:
        band = indicator_img.bandNames().getInfo()[0]
        areas = {}
        for c in [-1, 0, 1]:
            mask = indicator_img.eq(c).updateMask(water_mask)
            result = mask.multiply(ee.Image.pixelArea()).reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=aoi_geom,
                scale=SCALE,
                maxPixels=1e10
            )
            areas[c] = result.getInfo().get(band, 0)
        total = sum(areas.values())
        rows = []
        for c, name in [(-1,'Degraded'), (0,'Stable'), (1,'Improving')]:
            a_km2 = areas[c] / 1e6
            rows.append({
                'Class': name,
                'Area_km2': round(a_km2, 2),
                'Percent_%': round(100 * a_km2 / (total/1e6), 2) if total else 0
            })
        df = pd.DataFrame(rows)
    except Exception as e:
        print(f"⚠️ Warning: Could not compute statistics for '{label}' due to missing asset or EE exception.")
        # Return empty DataFrame matching the structure to prevent downstream failures
        df = pd.DataFrame(columns=['Class', 'Area_km2', 'Percent_%'])

    df.name = label
    return df

print("Computing area statistics … (may take 1–3 min per layer)")
df_prod   = compute_area_stats(productivity,  'Land Productivity')
df_lc     = compute_area_stats(lc_indicator,  'Land Cover')
df_soc    = compute_area_stats(soc_indicator, 'Soil Organic Carbon')
df_sdg    = compute_area_stats(sdg_15_3_1,    'SDG 15.3.1')
df_status = compute_area_stats(status_map,    'Status Map')

for df in [df_prod, df_lc, df_soc, df_sdg, df_status]:
    print(f"\n── {df.name} ──")
    if not df.empty:
        display(df)
    else:
        print("No data available (calculation skipped due to error)")

Computing area statistics … (may take 1–3 min per layer)
⚠️ Warning: Could not compute statistics for 'Land Productivity' due to missing asset or EE exception.
⚠️ Warning: Could not compute statistics for 'Soil Organic Carbon' due to missing asset or EE exception.
⚠️ Warning: Could not compute statistics for 'SDG 15.3.1' due to missing asset or EE exception.
⚠️ Warning: Could not compute statistics for 'Status Map' due to missing asset or EE exception.

── Land Productivity ──
No data available (calculation skipped due to error)

── Land Cover ──


,Class,Area_km2,Percent_%
0,Degraded,0.0,0
1,Stable,0.0,0
2,Improving,0.0,0



── Soil Organic Carbon ──
No data available (calculation skipped due to error)

── SDG 15.3.1 ──
No data available (calculation skipped due to error)

── Status Map ──
No data available (calculation skipped due to error)


## Cell 14 – Interactive Map (geemap)

In [19]:
PALETTE_3  = ['#d73027', '#fee08b', '#1a9850']   # Degraded / Stable / Improving
viz_params = dict(min=-1, max=1, palette=PALETTE_3)

Map = geemap.Map(center=[-1.85, 30.62], zoom=10)
Map.addLayer(AOI.style(color='white', fillColor='00000000', width=2),
             {}, 'Kayonza Boundary')

layers_to_add = [
    (productivity, 'Land Productivity'),
    (lc_indicator, 'Land Cover Change'),
    (soc_indicator, 'Soil Organic Carbon'),
    (sdg_15_3_1,    'SDG 15.3.1 (1OAO)'),
    (status_map,    'Status Map')
]

for img_obj, name in layers_to_add:
    try:
        # Force a light server-side evaluation check before adding to map
        img_obj.bandNames().getInfo()
        Map.addLayer(img_obj, viz_params, name)
        print(f"✓ Successfully added layer: {name}")
    except Exception:
        print(f"⚠️ Skipped layer: {name} (contains missing or invalid Earth Engine assets)")

Map.add_legend(title='SDG 15.3.1',
               legend_keys=['Degraded', 'Stable', 'Improving'],
               legend_colors=PALETTE_3)
Map

⚠️ Skipped layer: Land Productivity (contains missing or invalid Earth Engine assets)
✓ Successfully added layer: Land Cover Change
⚠️ Skipped layer: Soil Organic Carbon (contains missing or invalid Earth Engine assets)
⚠️ Skipped layer: SDG 15.3.1 (1OAO) (contains missing or invalid Earth Engine assets)
⚠️ Skipped layer: Status Map (contains missing or invalid Earth Engine assets)


Map(center=[-1.85, 30.62], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

## Cell 15 – Summary Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=False)
fig.suptitle('SDG 15.3.1 – Kayonza District, Rwanda\nLand Degradation Summary',
             fontsize=14, fontweight='bold')

datasets = [
    (df_prod, 'Land Productivity'),
    (df_lc,   'Land Cover'),
    (df_soc,  'Soil Organic Carbon'),
    (df_sdg,  'SDG 15.3.1 (Final)'),
]
colors = {'Degraded': '#d73027', 'Stable': '#fee08b', 'Improving': '#1a9850'}

for ax, (df, title) in zip(axes, datasets):
    bars = ax.bar(df['Class'], df['Percent_%'],
                  color=[colors[c] for c in df['Class']],
                  edgecolor='grey', linewidth=0.5)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('Area (%)')
    ax.set_ylim(0, 100)
    for bar, pct in zip(bars, df['Percent_%']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{pct}%', ha='center', va='bottom', fontsize=9)
    ax.tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.savefig('sdg_15_3_1_kayonza_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved ✓")

## Cell 16 – Export to Google Drive

All layers exported as GeoTIFF (UTM Zone 36S, EPSG:32736) to  
`My Drive / SDG_15_3_1_Kayonza`.  
Monitor progress at: https://code.earthengine.google.com/tasks

In [ ]:
def export_image(img, description, folder='SDG_15_3_1_Kayonza'):
    task = ee.batch.Export.image.toDrive(
        image=img.toInt8(),
        description=description,
        folder=folder,
        region=aoi_geom,
        scale=SCALE,
        crs='EPSG:32736',   # UTM Zone 36S – covers Rwanda
        maxPixels=1e10
    )
    task.start()
    print(f"  ▶ Export started: {description}  [task id: {task.id}]")

print("Submitting export tasks …")
export_image(productivity,  'Kayonza_LandProductivity')
export_image(lc_indicator,  'Kayonza_LandCover')
export_image(soc_indicator, 'Kayonza_SOC')
export_image(sdg_15_3_1,    'Kayonza_SDG_15_3_1_Final')
export_image(status_map,    'Kayonza_StatusMap')
print("\nAll tasks submitted ✓")
print("Monitor at: https://code.earthengine.google.com/tasks")

## Cell 17 – Final Results Summary

In [ ]:
total_km2 = df_sdg['Area_km2'].sum()
degraded  = df_sdg.loc[df_sdg['Class']=='Degraded',  'Area_km2'].values[0]
stable    = df_sdg.loc[df_sdg['Class']=='Stable',    'Area_km2'].values[0]
improving = df_sdg.loc[df_sdg['Class']=='Improving', 'Area_km2'].values[0]

print("=" * 60)
print("  SDG 15.3.1 FINAL RESULTS – KAYONZA DISTRICT, RWANDA")
print(f"  Baseline        : {BASELINE_START}–{BASELINE_END}")
print(f"  Reporting period: {TARGET_START}–{TARGET_END}")
print("=" * 60)
print(f"  Total land area analysed : {total_km2:.1f} km²")
print(f"  Degraded                 : {degraded:.1f} km²  ({100*degraded/total_km2:.1f} %)")
print(f"  Stable                   : {stable:.1f} km²  ({100*stable/total_km2:.1f} %)")
print(f"  Improving                : {improving:.1f} km²  ({100*improving/total_km2:.1f} %)")
print()
print("  ▶ SDG 15.3.1 = proportion of land degraded over total land area:")
print(f"    {100*degraded/total_km2:.2f} %")
print("=" * 60)